
#Ingeset sprints.json file

1. Read the file using spark dataframe reader API
2. Define and enforce schena (preserve the nested structure)
3. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
4. Write to bronze delta table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints/"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

### Step 1 - Read the JSON file using DataFrameReader API

In [0]:
sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType())
])

In [0]:
sprints_df = (
    spark.read.format('json')
    .schema(sprints_schema)
    .option('multiLine','true')
    .option('mode','FAILFAST')
    .load(source_file)
) 


### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)


### Step 3 - Write to Bronze Delta Table

In [0]:
(
    sprints_final_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
# display(spark.table(table_name))

In [0]:
# %sql
# SELECT season, COUNT(*) FROM formula1.bronze.sprints GROUP BY season ORDER BY season;